# Registar um pipeline no mlflow

Um pipeline não é mais que a composição de vários modelos/transformações

In [1]:
import mlflow

In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [3]:
ROOT_PATH = '../../data/'
SEED = 42
TARGET_COL = "Outcome"

## Definir a diretoria onde as experiências são guardadas

In [4]:
from pathlib import Path

uri = "http://localhost5000"

mlflow.set_tracking_uri(uri)

## Fazer set da experiência "Diabetes Prediction Experiment"

In [5]:
mlflow.set_experiment("Diabetes Prediction Experiment")

MlflowException: API request to http://localhost5000/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPConnectionPool(host='localhost5000', port=80): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=Diabetes+Prediction+Experiment (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x0000029550AA6D20>: Failed to resolve 'localhost5000' ([Errno 11001] getaddrinfo failed)"))

## Criar os datasets

In [ ]:
train_path = ROOT_PATH + 'diabetes_train.csv'
test_path = ROOT_PATH + 'diabetes_test.csv'

train_set = pd.read_csv(train_path)
test_set = pd.read_csv(test_path)

X_train = train_set.drop([TARGET_COL], axis = 1)
y_train = train_set[TARGET_COL]

X_test = test_set.drop([TARGET_COL], axis = 1)
y_test = test_set[TARGET_COL]

X_train.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,3,173,78,39,185,33.8,0.970,31
1,6,134,80,37,370,46.2,0.238,46
2,5,104,74,0,0,28.8,0.153,48
3,1,139,46,19,83,28.7,0.654,22
4,5,137,108,0,0,48.8,0.227,37


## Criar uma run

In [ ]:
run = mlflow.start_run(run_name="Linear Regression Run - C0.1 - pipeline")
RUN_ID = run.info.run_uuid
RUN_ID

'015fffc52ade44898203885e3df7e060'

## Guardar datasets, modelos, artefactos, métricas e parametros da run

In [ ]:
# guardarmos o dataset de treino e de teste associado à run
train_dataset = mlflow.data.from_pandas(train_set, source=train_path, targets=TARGET_COL, name="Diabetes Train Dataset")
test_dataset = mlflow.data.from_pandas(test_set, source=test_path, targets=TARGET_COL, name="Diabetes Test Dataset")
mlflow.log_input(train_dataset, context="train")
mlflow.log_input(test_dataset, context="test")

# Guardamos a seed utilizado como parametro
mlflow.log_param("seed", SEED)

c:\Users\User\miniconda3\envs\OML\Lib\site-packages\mlflow\data\dataset_source_registry.py:149: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for '../../data/diabetes_train.csv'. Exception: 
  return _dataset_source_registry.resolve(
c:\Users\User\miniconda3\envs\OML\Lib\site-packages\mlflow\data\dataset_source_registry.py:149: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
c:\Users\User\miniconda3\envs\OML\Lib\site-packages\mlflow\data\dataset_source_registry.py:149: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for '../../data/diabetes_test.csv'. Exception: 
  return _dataset_source_registry.resolve(
c:\Users\User\miniconda3\envs\OML\Lib\site-packages\mlflow\data\dataset_source_registry.py:149

42

In [ ]:
lr_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("logistic_regression", LogisticRegression(random_state=SEED, C=0.1))
])
lr_pipeline.fit(X_train, y_train)
mlflow.sklearn.log_model(lr_pipeline, artifact_path="lr_pipeline", registered_model_name="logistic_reg")
lr_pipeline

2025/03/01 11:32:22 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'logistic_reg'.
Created version '1' of model 'logistic_reg'.


Pipeline(steps=[('scaler', StandardScaler()),
                ('logistic_regression',
                 LogisticRegression(C=0.1, random_state=42))])

In [ ]:
params=lr_pipeline.get_params()

modified_params = {}
for k, v in params.items():
    new_key = k.replace("logistic_regression__", '')
    modified_params[new_key] = v

mlflow.log_params(modified_params)
modified_params

{'memory': None,
 'steps': [('scaler', StandardScaler()),
  ('logistic_regression', LogisticRegression(C=0.1, random_state=42))],
 'transform_input': None,
 'verbose': 0,
 'scaler': StandardScaler(),
 'logistic_regression': LogisticRegression(C=0.1, random_state=42),
 'scaler__copy': True,
 'scaler__with_mean': True,
 'scaler__with_std': True,
 'C': 0.1,
 'class_weight': None,
 'dual': False,
 'fit_intercept': True,
 'intercept_scaling': 1,
 'l1_ratio': None,
 'max_iter': 100,
 'multi_class': 'deprecated',
 'n_jobs': None,
 'penalty': 'l2',
 'random_state': 42,
 'solver': 'lbfgs',
 'tol': 0.0001,
 'warm_start': False}

In [ ]:
y_preds = lr_pipeline.predict(X_test)
acc = accuracy_score(y_test, y_preds)
mlflow.log_metric("accuracy", acc)
acc

0.7105263157894737

## Terminar a run

In [ ]:
mlflow.end_run()

## Consultar uma run já concluida

In [ ]:
run = mlflow.get_run(RUN_ID)

In [ ]:
run.data

<RunData: metrics={'accuracy': 0.7105263157894737}, params={'C': '0.1',
 'class_weight': 'None',
 'dual': 'False',
 'fit_intercept': 'True',
 'intercept_scaling': '1',
 'l1_ratio': 'None',
 'logistic_regression': 'LogisticRegression(C=0.1, random_state=42)',
 'max_iter': '100',
 'memory': 'None',
 'multi_class': 'deprecated',
 'n_jobs': 'None',
 'penalty': 'l2',
 'random_state': '42',
 'scaler': 'StandardScaler()',
 'scaler__copy': 'True',
 'scaler__with_mean': 'True',
 'scaler__with_std': 'True',
 'seed': '42',
 'solver': 'lbfgs',
 'steps': "[('scaler', StandardScaler()), ('logistic_regression', "
          'LogisticRegression(C=0.1, random_state=42))]',
 'tol': '0.0001',
 'transform_input': 'None',
 'verbose': '0',
 'warm_start': 'False'}, tags={'mlflow.log-model.history': '[{"run_id": "015fffc52ade44898203885e3df7e060", '
                             '"artifact_path": "lr_pipeline", '
                             '"utc_time_created": "2025-03-01 '
                             '11:32

## Ver a experiência na UI do mlflow

A UI do mlflow permite ver de forma visual todas as experiências criadas e permite por exemplo, comparar, filtar e ordenar, as runs dentro de uma experiência de forma visual.

Para correr a UI do mflow é necessário executar, na raiz deste projeto (pasta rumos) e tendo activo o ambiente utilizado neste projeto, o comando:

`mlflow ui --backend-store-uri ./mlruns`

**Nota:** O comando em cima irá iniciar a UI de mlflow na porta 5000. Caso queiram mudar esta porta devem acrescentar `--port <PORT>` ao comando (em que <PORT> deve ser substituido pela porta desejada). 

O comando acima não irá funcionar caso tenham tido alguns problemas no Windows com a instalação do mlflow. Caso tenham problemas, considerem instalar o `mlflow-ui`, ao invés do mlflow.

Após executarem este comando, vão poder ver a UI do mlflow no vosso browser acedendo a 

`http://127.0.0.1:5000`

(se tiverem alterado a porta em que o mlflow UI é iniciado então devem de alterar também aqui o 5000 por essa porta)

Na tab de `Experiments` podem explorar as experiências e runs que criaram.